# 05 — 2026 Tournament Predictions

*Spec section 10 (notebooks), driven by section 4 (tournament simulation) and section 3.6 / 11 (edge detection & outputs).*

This notebook produces the headline 2026 World Cup forecasts:

- **group-finish probabilities** (1st / 2nd / 3rd / 4th per team),
- **stage-reaching & win probabilities** (R32 -> R16 -> QF -> SF -> Final -> champion),
- **per-match H/D/A predictions** for the group stage,
- **market edges** vs. Polymarket / bookmaker implied probabilities (spec 3.6).

The Monte Carlo engine is `simulate_tournament` with a `StrengthModel` (attack/defense per
team) over the 2026 `TournamentStructure` (12 groups of 4). When a trained Dixon-Coles
artifact exists it is used directly; otherwise we fit a quick model on synthetic data so the
full pipeline runs. We keep `n_sims` small here for speed — production uses 100,000.

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from polymbappe.config import Settings
from polymbappe.eval.market import compute_edges
from polymbappe.simulate.structure import load_structure_2026
from polymbappe.simulate.tournament import (
    StrengthModel,
    TournamentStructure,
    simulate_tournament,
)

settings = Settings()
N_SIMS = 2_000  # production: 100_000 (spec 4.1)
rng = np.random.default_rng(settings.random_seed)

## Load the 2026 structure and a strength model

`load_structure_2026` reads the published draw from `configs/tournament_2026.yaml` when
present, falling back to a deterministic, clearly-labelled 48-team placeholder. For the
strength model we prefer a trained Dixon-Coles artifact (`polymbappe train`) via
`StrengthModel.from_dixon_coles`; if none exists we build a `StrengthModel` directly from
synthetic attack/defense strengths so the simulation still runs end to end.

In [ ]:
structure: TournamentStructure = load_structure_2026(settings)
teams = structure.teams
print(f"{len(structure.groups)} groups, {len(teams)} teams")
print("Group A:", structure.groups[next(iter(structure.groups))])


def _load_strength_model() -> tuple[StrengthModel, bool]:
    """Use the trained Dixon-Coles artifact if available; else synthesize strengths."""
    try:
        from polymbappe.features.context import HOSTS_2026
        from polymbappe.models.train import load_artifact

        dc = load_artifact("dixon_coles", settings)
        return StrengthModel.from_dixon_coles(dc, hosts=HOSTS_2026), True
    except Exception as exc:  # pragma: no cover - notebook resilience
        print(f"No trained artifact ({exc!r}); synthesizing a StrengthModel.")
        # Strengths track the placeholder Elo spread so favorites/underdogs are realistic.
        base = structure.elo or {t: 1500.0 for t in teams}
        center = float(np.mean(list(base.values())))
        attack = {t: (base[t] - center) / 400.0 for t in teams}
        defense = {t: -(base[t] - center) / 400.0 for t in teams}
        return StrengthModel(attack=attack, defense=defense, home_advantage=0.25), False


model, is_trained = _load_strength_model()
print(f"Strength model source: {'TRAINED Dixon-Coles' if is_trained else 'SYNTHETIC'}")

## Run the Monte Carlo simulation

`simulate_tournament` plays the full bracket each iteration: group stage with FIFA 2026
tiebreakers, best-third-place qualification, then R32->Final with extra time / penalties and
the correlated within-sim strength updates described in spec 4.1.

In [ ]:
result = simulate_tournament(structure, model, n_sims=N_SIMS, rng=rng)
stage_probs = result.stage_probabilities()
group_probs = result.group_probabilities()
print(f"Simulated {result.n_sims:,} tournaments.")
stage_probs.head(10)

## Win-probability leaderboard

The `champion` column is the trophy probability per team (spec 4.3 / dashboard page 1).

In [ ]:
top = stage_probs.sort("champion", descending=True).head(12)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top["team"][::-1], top["champion"][::-1], color="seagreen")
ax.set_xlabel("P(champion)")
ax.set_title(f"2026 trophy probability — top 12 ({N_SIMS:,} sims)")
plt.tight_layout()
plt.show()
top.select(["team", "R16", "QF", "SF", "FINAL", "champion"])

## Group-stage advancement

Per-team finish-position probabilities (spec 4.3). For one group we show the chance each
team finishes 1st / 2nd / 3rd / 4th — the top two advance directly and 3rd may qualify as a
best-third-placed team.

In [ ]:
first_group = next(iter(structure.groups))
members = structure.groups[first_group]
gp = group_probs.filter(pl.col("team").is_in(members))
gp = gp.with_columns((pl.col("finish_1") + pl.col("finish_2")).alias("advance_direct"))
print(f"Group {first_group}")
gp.sort("advance_direct", descending=True)

## Per-match H/D/A predictions (group stage)

For market-edge comparison we need per-match home/draw/away probabilities. The
`StrengthModel` exposes a Poisson score matrix per fixture (neutral venue at a World Cup),
from which H/D/A fall out directly. We build the predictions table for every scheduled group
match — this mirrors `match_predictions.parquet` (spec 11).

In [ ]:
_PAIRS = [(0, 1), (2, 3), (0, 2), (1, 3), (0, 3), (1, 2)]  # round-robin schedule


def match_hda(home: str, away: str) -> tuple[float, float, float]:
    matrix = model.score_matrix(home, away, neutral=True)
    home_win = float(np.tril(matrix, k=-1).sum())
    draw = float(np.trace(matrix))
    away_win = float(np.triu(matrix, k=1).sum())
    return home_win, draw, away_win


rows: list[dict] = []
for group, mem in structure.groups.items():
    for k, (i, j) in enumerate(_PAIRS):
        h, a = mem[i], mem[j]
        ph, pd, pa = match_hda(h, a)
        rows.append({
            "match_id": f"{group}-{k}", "group": group, "home_team": h, "away_team": a,
            "model_home": ph, "model_draw": pd, "model_away": pa,
        })

match_predictions = pl.DataFrame(rows)
print(f"{match_predictions.height} group-stage match predictions")
match_predictions.head()

## Market edges (spec 3.6)

`compute_edges` joins model H/D/A probabilities to market-implied probabilities and flags
outcomes where they diverge by more than the threshold (default 5pp), returning signed edge
in basis points and a full-Kelly stake fraction, sorted by magnitude.

Real market odds come from the ingested `market_odds` table; if absent we synthesize a
noisy market (model probabilities + perturbation) purely to demonstrate the edge workflow.
Genuine edges require the market-blind *edge pipeline* (spec 3.6) — here we illustrate the
mechanics with the calibration-pipeline probabilities.

In [ ]:
def _load_market(ids: list[str]) -> tuple[pl.DataFrame, bool]:
    try:
        from polymbappe.data.store import read_table, table_exists
        from polymbappe.data.tables import Table

        if table_exists(Table.MARKET_ODDS, settings):
            return read_table(Table.MARKET_ODDS, settings), True
    except Exception as exc:  # pragma: no cover - notebook resilience
        print(f"Market odds unavailable ({exc!r}); synthesizing a noisy market.")
    noise = rng.normal(0, 0.06, size=(len(ids), 3))
    base = match_predictions.select(["model_home", "model_draw", "model_away"]).to_numpy()
    perturbed = np.clip(base + noise, 1e-3, None)
    perturbed /= perturbed.sum(axis=1, keepdims=True)
    synthetic = pl.DataFrame({
        "match_id": ids,
        "home_win_prob": perturbed[:, 0],
        "draw_prob": perturbed[:, 1],
        "away_win_prob": perturbed[:, 2],
    })
    return synthetic, False


market, is_real_market = _load_market(match_predictions["match_id"].to_list())
edges = compute_edges(match_predictions, market, threshold=0.05)
print(f"Market source: {'REAL' if is_real_market else 'SYNTHETIC'} | {edges.height} edges flagged")
edges.head(10)

## Persisting outputs

Production runs write these to `data/outputs/` as parquet (spec 11):
`stage_probabilities.parquet`, `group_probabilities.parquet`, `match_predictions.parquet`,
`edges.parquet`. We show the write path but guard it so the notebook is side-effect-free by
default. Flip `WRITE_OUTPUTS = True` to persist.

In [ ]:
WRITE_OUTPUTS = False
if WRITE_OUTPUTS:
    out = settings.outputs_data_dir
    out.mkdir(parents=True, exist_ok=True)
    stage_probs.write_parquet(out / "stage_probabilities.parquet")
    group_probs.write_parquet(out / "group_probabilities.parquet")
    match_predictions.write_parquet(out / "match_predictions.parquet")
    edges.write_parquet(out / "edges.parquet")
    print(f"Wrote 4 artifacts to {out}")
else:
    print("WRITE_OUTPUTS is False — not persisting. Set True to write parquet artifacts.")

## Summary

- `simulate_tournament` over the 2026 `TournamentStructure` yields trophy, stage-reaching,
  and group-finish probabilities (spec 4.3).
- Per-match H/D/A predictions feed `compute_edges` for market-edge detection (spec 3.6).
- With trained artifacts and ingested market odds present, every cell runs on real inputs —
  raise `N_SIMS` to 100,000 and point at the edge pipeline for production forecasts.